# Init


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read from bronze table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

# Data transformations

## Renaming columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date" 
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trimming 


In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalization

In [0]:
df = (
    df
    .withColumn(
        "marital_status",
        F.when(F.upper(F.col("marital_status"))== "S", "Single")
         .when(F.upper(F.col("marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "gender",
        F.when(F.upper(F.col("gender"))== "F", "Female")
         .when(F.upper(F.col("gender")) == "M", "Male")
         .otherwise("n/a")
    )
)

## Sanity check of final DataFrame

In [0]:
df.limit(10).display()

# Write into silver table

In [0]:
(
    df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("silver.crm_customers")
)